# Complete the BM25 / dense BGE / fusion comparison

This notebook fills the missing verdict comparisons on the **official AVeriTeC development split**. It reuses the completed BM25, dense BGE-small, and BM25+BGE fusion rankings. It does not use the test split.

There are two result types:

1. **Fast controlled evidence swap (minutes):** keep the existing BM25-trained DistilBERT fixed and evaluate it with BM25, dense, or fusion development passages. Only the evidence changes. Dense and fusion rows are **diagnostics**, not matched train/test pipelines.
2. **Full matched runs (potentially days):** embed all training candidate pools with the *same* BGE model and settings used on development; fuse training rankings; then separately tune/refit DistilBERT on dense or fusion training evidence. Each claim and training checkpoint is saved on Drive, so interrupted runs can resume.

Run setup, checks, and the fast comparison first. For the long section, set RUN_FULL_MATCHED_TRAIN to True, then run its cells in order. The saved claim-only and human-annotated-answer controls are reused or regenerated if missing. Do not repeat them for each retriever: neither uses retrieved evidence.

Before opening this notebook in Colab, commit and push it and the updated dense runner. Use a GPU runtime. Keep paths, run names, and model settings unchanged when resuming.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice')
if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements/colab.txt'], check=True)

HF_CACHE = Path('/content/huggingface_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(HF_CACHE / 'sentence-transformers')
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, restart, then rerun setup.')
print('Repository:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name(0))
for name in ('run_dense_retrieval.py', 'run_hybrid_retrieval.py',
             'train_transformer_verifier.py', 'prepare_verifier_variant_rankings.py'):
    path = REPO_DIR / 'scripts' / name
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}; push the notebook and scripts, then pull in Colab.')
if '--repair-checkpoint' not in (REPO_DIR / 'scripts/run_dense_retrieval.py').read_text(encoding='utf-8'):
    raise RuntimeError('Dense runner is outdated; push the updated script and rerun setup.')

In [ ]:
import json
from src.utils.io import read_jsonl

DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed/averitec'
DEV_CLAIMS = PROCESSED / 'dev.jsonl'
TRAIN_CLAIMS = PROCESSED / 'train.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
TRAIN_BM25 = DATA_ROOT / 'outputs/bm25/train_bm25_top20_v2/train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl'
BM25_METRICS = DATA_ROOT / 'outputs/bm25/dev_bm25_top20_v2/dev_metrics.json'
PRIOR_ROOT = DATA_ROOT / 'outputs/initial_ml_solution_rerun_bs16'
DEV_DENSE_DIR = PRIOR_ROOT / 'dense/dev_bge_small_v15'
DEV_DENSE = DEV_DENSE_DIR / 'dev_rankings.jsonl'
DEV_DENSE_METRICS = DEV_DENSE_DIR / 'dev_metrics.json'
DEV_DENSE_CONFIG = DEV_DENSE_DIR / 'dense_run_config.json'
DEV_FUSION_DIR = PRIOR_ROOT / 'hybrid/dev_rrf60'
DEV_FUSION = DEV_FUSION_DIR / 'hybrid_rankings.jsonl'
DEV_FUSION_METRICS = DEV_FUSION_DIR / 'hybrid_metrics.json'
PRIOR_BM25_VERIFIER = PRIOR_ROOT / 'transformer/bm25_top3'
PRIOR_CONTROLS = DATA_ROOT / 'outputs/initial_ml_solution_followup_v1/transformer'
RUN_ROOT = DATA_ROOT / 'outputs/initial_ml_solution_dense_fusion_verifiers_v1'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

required = [TRAIN_CLAIMS, DEV_CLAIMS, DEV_QRELS, TRAIN_BM25, DEV_BM25,
            BM25_METRICS, DEV_DENSE, DEV_DENSE_METRICS, DEV_DENSE_CONFIG,
            DEV_FUSION, DEV_FUSION_METRICS]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing prior Drive inputs:\n' + '\n'.join(missing))

def ranking_ids(path):
    rows = read_jsonl(path)
    ids = [str(row['claim_id']) for row in rows]
    if len(ids) != len(set(ids)) or any(not isinstance(row.get('retrieved'), list) for row in rows):
        raise ValueError(f'Duplicate IDs or malformed rankings: {path}')
    return set(ids)

train_ids = {str(row['claim_id']) for row in read_jsonl(TRAIN_CLAIMS)}
dev_ids = {str(row['claim_id']) for row in read_jsonl(DEV_CLAIMS)}
if len(train_ids) != 3067 or len(dev_ids) != 500:
    raise ValueError(f'Unexpected split sizes: train={len(train_ids)}, dev={len(dev_ids)}')
if ranking_ids(TRAIN_BM25) != train_ids:
    raise ValueError('BM25 training rankings do not cover the official usable training claims.')
for path in (DEV_BM25, DEV_DENSE, DEV_FUSION):
    if ranking_ids(path) != dev_ids:
        raise ValueError(f'Development rankings do not cover the same 500 claims: {path}')

DENSE_DEV_REPORT = json.loads(DEV_DENSE_METRICS.read_text(encoding='utf-8'))
DENSE_DEV_CONFIG = json.loads(DEV_DENSE_CONFIG.read_text(encoding='utf-8'))
FUSION_DEV_REPORT = json.loads(DEV_FUSION_METRICS.read_text(encoding='utf-8'))
DENSE_MODEL = 'BAAI/bge-small-en-v1.5'
DENSE_PREFIX = 'Represent this sentence for searching relevant passages: '
DENSE_REV = DENSE_DEV_REPORT.get('resolved_revision')
if (DENSE_DEV_REPORT.get('model') != DENSE_MODEL or
    DENSE_DEV_REPORT.get('query_prefix') != DENSE_PREFIX or
    DENSE_DEV_REPORT.get('ranking_k') != 20 or
    DENSE_DEV_REPORT.get('batch_size') != 16 or
    DENSE_DEV_REPORT.get('max_seq_length') != 384 or
    not DENSE_REV):
    raise ValueError('Saved dense dev settings differ; do not combine unmatched runs.')
if DENSE_DEV_CONFIG.get('max_words') != 160 or DENSE_DEV_CONFIG.get('overlap_words') != 40:
    raise ValueError('Saved dense dev chunking differs from 160/40-word train chunking.')
if FUSION_DEV_REPORT.get('rrf_constant') != 60 or FUSION_DEV_REPORT.get('ranking_k') != 20:
    raise ValueError('Saved fusion dev settings differ from RRF-60/top-20.')
if Path(FUSION_DEV_REPORT['bm25_rankings']) != DEV_BM25 or Path(FUSION_DEV_REPORT['dense_rankings']) != DEV_DENSE:
    raise ValueError('Saved fusion dev rankings were built from different inputs.')
DISTILBERT_REV = '12040accade4e8a0f71eabdb258fecc2e7e948be'
prior_config_path = PRIOR_BM25_VERIFIER / 'transformer_run_config.json'
if prior_config_path.is_file():
    prior_revision = json.loads(prior_config_path.read_text(encoding='utf-8')).get('resolved_model_revision')
    if prior_revision and prior_revision != DISTILBERT_REV:
        raise ValueError(f'Prior DistilBERT revision differs: {prior_revision}')
print('Input coverage passed: 3,067 train / 500 dev claims.')
print('BGE revision:', DENSE_REV)
print('Output root:', RUN_ROOT)

## Direct-run and resume helpers

Experiment scripts execute in the notebook kernel, so their progress bars and full tracebacks stay visible. Reconnect, rerun setup/checks/helpers, and rerun the interrupted cell. Do not rename an in-progress run.

In [ ]:
import gc
import runpy
import time

def complete(paths):
    for path in map(Path, paths):
        if not path.is_file() or path.stat().st_size == 0:
            return False
        if path.suffix == '.json':
            try:
                json.loads(path.read_text(encoding='utf-8'))
            except (OSError, ValueError):
                return False
    return True

def run_script(label, script_name, args, outputs=None):
    if outputs and complete(outputs):
        print(f'{label}: complete; reusing saved artifacts.')
        return
    script = (REPO_DIR / 'scripts' / script_name).resolve()
    if not script.is_relative_to(REPO_DIR.resolve()) or not script.is_file():
        raise FileNotFoundError(script)
    old_argv = sys.argv[:]
    sys.argv = [str(script), *map(str, args)]
    print(f'\n=== {label} ===\nRunner: {script}\nArguments: {args}', flush=True)
    started = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    except KeyboardInterrupt:
        print('Interrupted; rerun this cell unchanged to resume.', flush=True)
        raise
    finally:
        sys.argv = old_argv
        gc.collect()
        torch.cuda.empty_cache()
    if outputs and not complete(outputs):
        raise RuntimeError(f'{label} did not produce complete artifacts: {outputs}')
    print(f'{label} finished in {(time.monotonic()-started)/60:.1f} minutes.')

def verifier_args(name, train_rankings, dev_rankings, output_root):
    return [
        '--data-root', DATA_ROOT, '--train-rankings', train_rankings,
        '--eval-rankings', dev_rankings, '--output-dir', output_root, '--run-name', name,
        '--model', 'distilbert-base-uncased', '--revision', DISTILBERT_REV,
        '--max-length', '512', '--claim-token-budget', '96', '--evidence-token-budget', '110',
        '--validation-fraction', '0.10', '--learning-rates', '2e-5,5e-5',
        '--epoch-grid', '2,3', '--batch-size', '8',
        '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100',
        '--seed', '718', '--resume',
    ]

def verifier_outputs(path):
    return [path / 'final_dev_metrics.json', path / 'final_dev_predictions.jsonl',
            path / 'transformer_manifest.json']

## Reuse or regenerate the BM25 verifier and controls

The fast evidence swap needs the completed BM25-trained DistilBERT. Claim-only and gold-answer controls are included in the final comparison. If their previous artifacts are missing, this cell regenerates them with the same training settings. Gold answers are oracle-only and never enter a deployable retrieval pipeline.

In [ ]:
def has_final_model(directory):
    model_dir = directory / 'final_model'
    weights = [*model_dir.glob('*.safetensors'), *model_dir.glob('pytorch_model*.bin')]
    return complete([*verifier_outputs(directory), model_dir / 'config.json',
                     directory / 'transformer_run_config.json']) and bool(weights)

BM25_VERIFIER = PRIOR_BM25_VERIFIER
if not has_final_model(BM25_VERIFIER):
    BM25_VERIFIER = RUN_ROOT / 'transformer/bm25_top3'
    run_script('DistilBERT / BM25 top 3', 'train_transformer_verifier.py',
               verifier_args('bm25_top3', TRAIN_BM25, DEV_BM25, RUN_ROOT / 'transformer'),
               verifier_outputs(BM25_VERIFIER))
if not has_final_model(BM25_VERIFIER):
    raise RuntimeError('BM25 verifier has no complete final model; inspect its Drive run.')

CONTROL_DIRS = {}
for mode, prior_name in (('claim_only', 'claim_only'), ('gold', 'gold_oracle')):
    prior = PRIOR_CONTROLS / prior_name
    if complete(verifier_outputs(prior)):
        CONTROL_DIRS[mode] = prior
        print(f'{mode}: reusing', prior)
        continue
    variant = RUN_ROOT / 'variants' / mode
    run_script(f'Prepare {mode} control evidence', 'prepare_verifier_variant_rankings.py',
               ['--data-root', DATA_ROOT, '--output-dir', variant, '--mode', mode],
               [variant / 'train_rankings.jsonl', variant / 'dev_rankings.jsonl',
                variant / 'variant_manifest.json'])
    output = RUN_ROOT / 'transformer' / prior_name
    run_script(f'DistilBERT / {mode} control', 'train_transformer_verifier.py',
               verifier_args(prior_name, variant / 'train_rankings.jsonl',
                             variant / 'dev_rankings.jsonl', RUN_ROOT / 'transformer'),
               verifier_outputs(output))
    CONTROL_DIRS[mode] = output
print('BM25 verifier:', BM25_VERIFIER)
print('Control verifiers:', CONTROL_DIRS)

## Fast diagnostic — one frozen verifier, three retrievers

The BM25-trained final DistilBERT is **not retrained** here. We already have its BM25-development predictions; this cell evaluates the same weights on dense and fusion top-three passages. The comparison isolates the effect of changing development evidence, but the new evidence distribution was not used in training. Report these as evidence-swap diagnostics, not as matched dense/fusion models. Per-claim predictions are checkpointed on Drive.

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scripts.run_evidence_ablation import load_partial_predictions
from scripts.train_tfidf_logreg import attach_retrieved_evidence
from scripts.train_transformer_verifier import compose_text
from src.data.averitec import CANONICAL_LABELS
from src.evaluation.classification import evaluate_predictions
from src.utils.io import write_json, write_jsonl
from src.utils.runs import sha256_file

SWAP_ROOT = RUN_ROOT / 'frozen_bm25_trained_evidence_swap'
model_settings = json.loads((BM25_VERIFIER / 'transformer_run_config.json').read_text(encoding='utf-8'))
model_dir = BM25_VERIFIER / 'final_model'
if not (model_dir / 'config.json').is_file():
    raise FileNotFoundError(f'Missing final BM25 model: {model_dir}')
weight_files = sorted((*model_dir.glob('*.safetensors'), *model_dir.glob('pytorch_model*.bin')))
if not weight_files:
    raise FileNotFoundError(f'Missing final BM25 model weights: {model_dir}')
weight_hashes = {path.name: sha256_file(path) for path in weight_files}
tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to('cuda').eval()
if [str(model.config.id2label[i]) for i in range(4)] != list(CANONICAL_LABELS):
    raise ValueError('Verifier label order differs from the evaluation labels.')

def frozen_evidence_eval(name, rankings):
    out = SWAP_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    predictions_path = out / 'predictions.jsonl'
    metrics_path = out / 'metrics.json'
    config_path = out / 'run_config.json'
    config = {
        'evaluation_type': 'frozen_BM25_trained_verifier_evidence_swap',
        'dev_claims_sha256': sha256_file(DEV_CLAIMS),
        'dev_rankings_sha256': sha256_file(rankings),
        'model_run_config_sha256': sha256_file(BM25_VERIFIER / 'transformer_run_config.json'),
        'model_config_sha256': sha256_file(model_dir / 'config.json'),
        'model_weights_sha256': weight_hashes,
        'model_dir': str(model_dir.resolve()),
        'train_evidence': str(TRAIN_BM25),
        'eval_evidence': str(rankings),
        'max_evidence': 3,
    }
    prior = json.loads(config_path.read_text(encoding='utf-8')) if config_path.is_file() else None
    if prior is not None and prior != config:
        raise ValueError(f'Inputs changed for {out}; use a new run name.')
    if complete([predictions_path, metrics_path]):
        print(name, ': complete; reusing saved predictions.')
        return
    write_json(config, config_path)
    records = attach_retrieved_evidence(read_jsonl(DEV_CLAIMS), rankings)
    if len(records) != 500 or {str(row['claim_id']) for row in records} != dev_ids:
        raise ValueError('Evidence-swap evaluation must cover the same 500 dev claims.')
    partial_path = out / 'predictions.partial.jsonl'
    completed = load_partial_predictions(partial_path, dev_ids)
    remaining = [row for row in records if str(row['claim_id']) not in completed]
    with tqdm(total=500, initial=len(completed), desc=f'Frozen verifier / {name}', unit='claim') as bar:
        for start in range(0, len(remaining), 16):
            batch = remaining[start:start+16]
            texts = [compose_text(row, tokenizer,
                                  int(model_settings['claim_token_budget']),
                                  int(model_settings['evidence_token_budget_per_passage'])) for row in batch]
            encoded = tokenizer(texts, padding=True, truncation=True,
                                max_length=int(model_settings['max_length']),
                                return_tensors='pt')
            encoded = {key: value.to('cuda') for key, value in encoded.items()}
            with torch.inference_mode():
                probs = torch.softmax(model(**encoded).logits.float(), dim=-1).cpu().tolist()
            pending = []
            for row, values in zip(batch, probs, strict=True):
                claim_id = str(row['claim_id'])
                evidence = row.get('retrieved_evidence') or []
                prediction = {
                    'claim_id': claim_id, 'true_label': str(row['label']),
                    'predicted_label': CANONICAL_LABELS[max(range(4), key=values.__getitem__)],
                    'probabilities': {label: float(values[i]) for i, label in enumerate(CANONICAL_LABELS)},
                    'evidence_item_count': min(3, len(evidence)),
                    'evidence_passage_ids': [str(item['passage_id']) for item in evidence[:3]],
                }
                completed[claim_id] = prediction
                pending.append(prediction)
            with partial_path.open('a', encoding='utf-8', newline='\n') as handle:
                for row in pending:
                    handle.write(json.dumps(row, ensure_ascii=False) + '\n')
                handle.flush()
            bar.update(len(pending))
    if len(completed) != 500:
        raise RuntimeError(f'Only {len(completed)} / 500 predictions completed.')
    ordered = [completed[str(row['claim_id'])] for row in records]
    temporary = predictions_path.with_name(predictions_path.name + '.tmp')
    write_jsonl(ordered, temporary)
    temporary.replace(predictions_path)
    report = {**evaluate_predictions(ordered), **config, 'claim_count': 500,
              'matched_train_eval_evidence_pipeline': False}
    temporary_metrics = metrics_path.with_name(metrics_path.name + '.tmp')
    write_json(report, temporary_metrics)
    temporary_metrics.replace(metrics_path)
    print(name, {key: report[key] for key in ('macro_f1', 'accuracy', 'decision_error_rates')})

for label, path in (('dense', DEV_DENSE), ('fusion', DEV_FUSION)):
    frozen_evidence_eval(label, path)
del model
torch.cuda.empty_cache()

## Optional full matched train/evaluation

Set RUN_FULL_MATCHED_TRAIN to True only when ready for the long job. On the previous development run, full-pool BGE encoding took roughly 12 hours across interruptions; 3,067 training claims may take substantially longer. The exact model revision, 160/40-word chunks, 384-token limit, and top-20 ranking depth match the completed development run. Each archive shard and completed claim survives Colab disconnects. The updated runner can back up and repair an interrupted final JSONL line or identical duplicate; it refuses conflicting duplicates.

If the long run does not finish by the deadline, submit the completed fast diagnostic with its training/evaluation mismatch stated plainly. Do not report it as a matched dense or fusion classifier.

In [ ]:
RUN_FULL_MATCHED_TRAIN = False  # Set True to run all long cells below.
TRAIN_ARCHIVES = [
    DATA_ROOT / 'raw/averitec/train_0_999.zip',
    DATA_ROOT / 'raw/averitec/train_1000_1999.zip',
    DATA_ROOT / 'raw/averitec/train_2000_3067.zip',
]
DENSE_TRAIN_DIR = RUN_ROOT / 'dense/train_bge_small_v15_matched'
TRAIN_DENSE = DENSE_TRAIN_DIR / 'train_rankings.jsonl'
TRAIN_DENSE_METRICS = DENSE_TRAIN_DIR / 'train_metrics.json'
TRAIN_DENSE_MANIFEST = DENSE_TRAIN_DIR / 'train_manifest.json'
FUSION_TRAIN_DIR = RUN_ROOT / 'fusion/train_rrf60_matched'
TRAIN_FUSION = FUSION_TRAIN_DIR / 'hybrid_rankings.jsonl'
TRAIN_FUSION_METRICS = FUSION_TRAIN_DIR / 'hybrid_metrics.json'
TRAIN_FUSION_MANIFEST = FUSION_TRAIN_DIR / 'hybrid_manifest.json'
DENSE_VERIFIER = RUN_ROOT / 'transformer/dense_top3_matched'
FUSION_VERIFIER = RUN_ROOT / 'transformer/fusion_top3_matched'
if RUN_FULL_MATCHED_TRAIN:
    missing_archives = [str(p) for p in TRAIN_ARCHIVES if not p.is_file()]
    if missing_archives:
        raise FileNotFoundError('Missing official train archives:\n' + '\n'.join(missing_archives))
print('Full matched training enabled:', RUN_FULL_MATCHED_TRAIN)

In [ ]:
if not RUN_FULL_MATCHED_TRAIN:
    print('Skipped long training retrieval. Set RUN_FULL_MATCHED_TRAIN=True, then rerun from this cell.')
elif complete([TRAIN_DENSE, TRAIN_DENSE_METRICS, TRAIN_DENSE_MANIFEST]):
    print('Full dense training rankings already finalized:', TRAIN_DENSE)
else:
    for archive in TRAIN_ARCHIVES:
        args = [
            '--split', 'train', '--data-root', DATA_ROOT, '--archive', archive,
            '--output-dir', RUN_ROOT / 'dense', '--run-name', DENSE_TRAIN_DIR.name,
            '--model', DENSE_MODEL, '--revision', DENSE_REV,
            '--query-prefix', DENSE_PREFIX, '--device', 'cuda',
            '--ranking-k', '20', '--metric-k', '3',
            '--batch-size', '16', '--max-seq-length', '384',
            '--max-words', '160', '--overlap-words', '40',
            '--checkpoint-every', '1', '--resume', '--repair-checkpoint',
        ]
        run_script(f'Full-pool BGE train retrieval / {archive.name}',
                   'run_dense_retrieval.py', args)
        progress_file = DENSE_TRAIN_DIR / 'train_progress.json'
        if progress_file.is_file():
            progress_data = json.loads(progress_file.read_text(encoding='utf-8'))
            print('Claims completed in this invocation:',
                  progress_data.get('processed_this_invocation_count'))
    print('All archive passes returned; finalize in the next cell.')

In [ ]:
if not RUN_FULL_MATCHED_TRAIN:
    print('Skipped dense train finalization.')
else:
    run_script('Finalize full-pool BGE train rankings', 'run_dense_retrieval.py', [
        '--split', 'train', '--data-root', DATA_ROOT,
        '--output-dir', RUN_ROOT / 'dense', '--run-name', DENSE_TRAIN_DIR.name,
        '--model', DENSE_MODEL, '--revision', DENSE_REV,
        '--query-prefix', DENSE_PREFIX, '--ranking-k', '20', '--metric-k', '3',
        '--batch-size', '16', '--max-seq-length', '384',
        '--max-words', '160', '--overlap-words', '40',
        '--resume', '--repair-checkpoint', '--finalize',
    ], [TRAIN_DENSE, TRAIN_DENSE_METRICS, TRAIN_DENSE_MANIFEST])
    if ranking_ids(TRAIN_DENSE) != train_ids:
        raise ValueError('Finalized dense train rankings do not match 3,067 train claims.')
    print('Dense train ranking complete:', TRAIN_DENSE)

In [ ]:
if not RUN_FULL_MATCHED_TRAIN:
    print('Skipped train fusion.')
else:
    if not complete([TRAIN_DENSE, TRAIN_DENSE_METRICS, TRAIN_DENSE_MANIFEST]):
        raise FileNotFoundError('Finalize dense train rankings first.')
    run_script('BM25 + BGE train fusion', 'run_hybrid_retrieval.py', [
        '--bm25-rankings', TRAIN_BM25, '--dense-rankings', TRAIN_DENSE,
        '--output-dir', RUN_ROOT / 'fusion', '--run-name', FUSION_TRAIN_DIR.name,
        '--rrf-constant', '60', '--ranking-k', '20', '--metric-k', '3',
    ], [TRAIN_FUSION, TRAIN_FUSION_METRICS, TRAIN_FUSION_MANIFEST])
    if ranking_ids(TRAIN_FUSION) != train_ids:
        raise ValueError('Fused train rankings do not match 3,067 train claims.')

In [ ]:
if not RUN_FULL_MATCHED_TRAIN:
    print('Skipped matched dense-evidence verifier.')
else:
    if not complete([TRAIN_DENSE, TRAIN_DENSE_METRICS, TRAIN_DENSE_MANIFEST]):
        raise FileNotFoundError('Finalize dense train rankings first.')
    run_script('DistilBERT trained/evaluated with full-pool BGE top 3',
               'train_transformer_verifier.py',
               verifier_args(DENSE_VERIFIER.name, TRAIN_DENSE, DEV_DENSE,
                             RUN_ROOT / 'transformer'),
               verifier_outputs(DENSE_VERIFIER))

In [ ]:
if not RUN_FULL_MATCHED_TRAIN:
    print('Skipped matched fusion-evidence verifier.')
else:
    if not complete([TRAIN_FUSION, TRAIN_FUSION_METRICS, TRAIN_FUSION_MANIFEST]):
        raise FileNotFoundError('Create train fusion rankings first.')
    run_script('DistilBERT trained/evaluated with BM25+BGE fusion top 3',
               'train_transformer_verifier.py',
               verifier_args(FUSION_VERIFIER.name, TRAIN_FUSION, DEV_FUSION,
                             RUN_ROOT / 'transformer'),
               verifier_outputs(FUSION_VERIFIER))

## Results for the report

The final table distinguishes prior baselines, controls, frozen evidence swaps, and genuinely matched evidence pipelines. Use the matched rows for an end-to-end comparison only after both training runs complete. The untouched test split is never loaded.

In [ ]:
import csv
from IPython.display import Markdown, display

def metric(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else None

retrieval_sources = [
    ('BM25', BM25_METRICS),
    ('BGE-small dense', DEV_DENSE_METRICS),
    ('BM25 + BGE fusion', DEV_FUSION_METRICS),
]
retrieval_rows = []
for name, path in retrieval_sources:
    result = metric(path)
    if result is None:
        continue
    retrieval_rows.append({
        'retriever': name,
        'URL_proxy_hit_at_3': result.get('evidence_hit_at_3'),
        'MRR_over_saved_top_20': result.get('mrr'),
        'judged_claims': result.get('judged_claim_count'),
        'metrics_file': str(path),
    })

QWEN_METRICS = DATA_ROOT / 'outputs/local_llm/dev_qwen3_4b_bm25_top3_v2/classify_top3_metrics.json'
LOGISTIC_METRICS = DATA_ROOT / 'outputs/tfidf_logreg/retrieved_train_to_dev_v2/retrieved_train_to_dev_metrics.json'
verdict_sources = [
    ('Qwen + BM25 (prior baseline)', 'prior baseline', QWEN_METRICS),
    ('TF-IDF/logistic + BM25 (prior baseline)', 'prior baseline', LOGISTIC_METRICS),
    ('DistilBERT + BM25 top 3', 'matched', BM25_VERIFIER / 'final_dev_metrics.json'),
    ('DistilBERT claim only', 'no-evidence control', CONTROL_DIRS['claim_only'] / 'final_dev_metrics.json'),
    ('DistilBERT annotated answers', 'oracle control', CONTROL_DIRS['gold'] / 'final_dev_metrics.json'),
    ('Frozen BM25-trained DistilBERT + BGE dev evidence', 'evidence-swap diagnostic', SWAP_ROOT / 'dense/metrics.json'),
    ('Frozen BM25-trained DistilBERT + fusion dev evidence', 'evidence-swap diagnostic', SWAP_ROOT / 'fusion/metrics.json'),
    ('DistilBERT + BGE train/dev top 3', 'matched', DENSE_VERIFIER / 'final_dev_metrics.json'),
    ('DistilBERT + fusion train/dev top 3', 'matched', FUSION_VERIFIER / 'final_dev_metrics.json'),
]
verdict_rows = []
for name, role, path in verdict_sources:
    result = metric(path)
    errors = (result or {}).get('decision_error_rates') or {}
    verdict_rows.append({
        'system': name, 'role': role,
        'macro_f1': (result or {}).get('macro_f1'),
        'accuracy': (result or {}).get('accuracy'),
        'false_reassurance_count': errors.get('false_reassurance_count'),
        'false_reassurance_rate_per_500': errors.get('false_reassurance_rate'),
        'false_alarm_count': errors.get('false_alarm_count'),
        'false_alarm_rate_per_500': errors.get('false_alarm_rate'),
        'metrics_file': str(path) if result else 'NOT RUN',
    })
def show_table(rows, columns):
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join('---' for _ in columns) + ' |']
    for row in rows:
        values = [row.get(column) for column in columns]
        lines.append('| ' + ' | '.join('—' if value is None else f'{value:.4f}' if isinstance(value, float) else str(value) for value in values) + ' |')
    display(Markdown('\n'.join(lines)))

print('RETRIEVAL: same 386 judged dev claims')
show_table(retrieval_rows, ['retriever', 'URL_proxy_hit_at_3', 'MRR_over_saved_top_20'])
print('VERDICTS: same 500 dev claims; diagnostic/oracle rows are not deployable')
show_table(verdict_rows, ['system', 'role', 'macro_f1', 'accuracy', 'false_reassurance_count', 'false_alarm_count'])
summary = RUN_ROOT / 'comparison_summary'
summary.mkdir(parents=True, exist_ok=True)
for filename, rows in (('retrieval.csv', retrieval_rows), ('verdicts.csv', verdict_rows)):
    with (summary / filename).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print('Saved summary CSVs:', summary)
if not complete([DENSE_VERIFIER / 'final_dev_metrics.json',
                 FUSION_VERIFIER / 'final_dev_metrics.json']):
    print('Matched dense/fusion classifier results are incomplete. Do not substitute frozen evidence-swap rows in the report.')